# 04 — Gold AI-Ready Analytics

Engineers analytics features, validates the Gold dataset, creates summary tables, and records experiment metrics.

## Publication notes

- This public notebook contains no credentials, tokens, tenant IDs, subscription IDs, or workspace URLs.
- Notebook outputs were removed to avoid publishing source records or environment metadata.
- Configure the catalog, schema, and source data location for your own Azure environment before execution.
- Run notebooks in numerical order.


In [ ]:
from pyspark.sql import functions as F
import uuid
import time

CATALOG = "dbx_niw_analytics"
SCHEMA = "ztlf"

SILVER_TABLE = f"{CATALOG}.{SCHEMA}.silver_customer_churn"
GOLD_TABLE = f"{CATALOG}.{SCHEMA}.gold_customer_churn"
GOLD_SUMMARY_TABLE = f"{CATALOG}.{SCHEMA}.gold_churn_summary"
METRICS_TABLE = f"{CATALOG}.{SCHEMA}.experiment_metrics"

GOLD_RUN_ID = str(uuid.uuid4())

print(f"Gold run ID: {GOLD_RUN_ID}")
print(f"Source table: {SILVER_TABLE}")

In [ ]:
silver_df = spark.table(SILVER_TABLE)

silver_row_count = silver_df.count()
silver_column_count = len(silver_df.columns)

print(f"Silver rows: {silver_row_count}")
print(f"Silver columns: {silver_column_count}")

display(silver_df.limit(20))

In [ ]:
silver_validation = silver_df.agg(
    F.count("*").alias("total_rows"),
    F.countDistinct("CustomerId").alias("distinct_customer_ids"),

    F.sum(
        F.col("CustomerId").isNull().cast("int")
    ).alias("null_customer_ids"),

    F.sum(
        F.col("EstimatedSalary").isNull().cast("int")
    ).alias("null_salaries"),

    F.sum(
        (~F.col("Age").between(18, 100)).cast("int")
    ).alias("invalid_ages"),

    F.sum(
        (F.col("Balance") < 0).cast("int")
    ).alias("negative_balances"),

    F.sum(
        (~F.col("Geography").isin("France", "Germany", "Spain")).cast("int")
    ).alias("invalid_geographies"),

    F.sum(
        (~F.col("Gender").isin("Male", "Female")).cast("int")
    ).alias("invalid_genders")
)

display(silver_validation)

In [ ]:
gold_df = (
    silver_df
    .withColumn(
        "AgeGroup",
        F.when(F.col("Age") < 30, "18-29")
         .when(F.col("Age") < 40, "30-39")
         .when(F.col("Age") < 50, "40-49")
         .when(F.col("Age") < 60, "50-59")
         .otherwise("60+")
    )
    .withColumn(
        "GenderCode",
        F.when(F.col("Gender") == "Male", F.lit(1))
         .otherwise(F.lit(0))
    )
)

In [ ]:
gold_df = (
    gold_df
    .withColumn(
        "CreditScoreBand",
        F.when(F.col("CreditScore") < 580, "Poor")
         .when(F.col("CreditScore") < 670, "Fair")
         .when(F.col("CreditScore") < 740, "Good")
         .when(F.col("CreditScore") < 800, "Very Good")
         .otherwise("Exceptional")
    )
    .withColumn(
        "LowCreditFlag",
        F.when(F.col("CreditScore") < 580, F.lit(1))
         .otherwise(F.lit(0))
    )
)

In [ ]:
gold_df = (
    gold_df
    .withColumn(
        "BalanceBand",
        F.when(F.col("Balance") == 0, "Zero Balance")
         .when(F.col("Balance") < 50000, "Low")
         .when(F.col("Balance") < 100000, "Medium")
         .when(F.col("Balance") < 150000, "High")
         .otherwise("Very High")
    )
    .withColumn(
        "SalaryBand",
        F.when(F.col("EstimatedSalary") < 50000, "Low")
         .when(F.col("EstimatedSalary") < 100000, "Medium")
         .when(F.col("EstimatedSalary") < 150000, "High")
         .otherwise("Very High")
    )
    .withColumn(
        "BalanceToSalaryRatio",
        F.when(
            F.col("EstimatedSalary") > 0,
            F.round(
                F.col("Balance") / F.col("EstimatedSalary"),
                4
            )
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "ZeroBalanceFlag",
        F.when(F.col("Balance") == 0, F.lit(1))
         .otherwise(F.lit(0))
    )
)

In [ ]:
gold_df = (
    gold_df
    .withColumn(
        "TenureGroup",
        F.when(F.col("Tenure") <= 2, "New")
         .when(F.col("Tenure") <= 5, "Established")
         .when(F.col("Tenure") <= 8, "Long-Term")
         .otherwise("Highly Tenured")
    )
    .withColumn(
        "ProductEngagementLevel",
        F.when(F.col("NumOfProducts") == 1, "Single Product")
         .when(F.col("NumOfProducts") == 2, "Multi Product")
         .otherwise("Highly Engaged")
    )
    .withColumn(
        "EngagementScore",
        (
            F.col("IsActiveMember").cast("int")
            + F.col("HasCrCard").cast("int")
            + F.when(F.col("NumOfProducts") >= 2, 1).otherwise(0)
        )
    )
    .withColumn(
        "InactiveMemberFlag",
        F.when(F.col("IsActiveMember") == 0, F.lit(1))
         .otherwise(F.lit(0))
    )
)

In [ ]:
gold_df = (
    gold_df
    .withColumn(
        "CustomerValueScore",
        F.round(
            (
                F.col("Balance") * F.lit(0.40)
                + F.col("EstimatedSalary") * F.lit(0.20)
                + F.col("CreditScore") * F.lit(50.0)
                + F.col("Tenure") * F.lit(1000.0)
                + F.col("NumOfProducts") * F.lit(5000.0)
            ),
            2
        )
    )
    .withColumn(
        "CustomerValueSegment",
        F.when(
            F.col("CustomerValueScore") < 70000,
            "Standard"
        )
        .when(
            F.col("CustomerValueScore") < 120000,
            "Valuable"
        )
        .otherwise("High Value")
    )
)

In [ ]:
gold_df = (
    gold_df
    .withColumn(
        "BehavioralRiskScore",
        (
            F.col("InactiveMemberFlag") * 2
            + F.col("ZeroBalanceFlag")
            + F.col("LowCreditFlag")
            + F.when(F.col("NumOfProducts") == 1, 1).otherwise(0)
            + F.when(F.col("Age") >= 50, 1).otherwise(0)
        )
    )
    .withColumn(
        "BehavioralRiskBand",
        F.when(F.col("BehavioralRiskScore") <= 1, "Low")
         .when(F.col("BehavioralRiskScore") <= 3, "Medium")
         .otherwise("High")
    )
)

In [ ]:
gold_df = (
    gold_df
    .withColumn(
        "GeographyCode",
        F.when(F.col("Geography") == "France", F.lit(0))
         .when(F.col("Geography") == "Germany", F.lit(1))
         .when(F.col("Geography") == "Spain", F.lit(2))
    )
)

In [ ]:
gold_df = (
    gold_df
    .withColumn("_gold_run_id", F.lit(GOLD_RUN_ID))
    .withColumn("_gold_processed_at", F.current_timestamp())
    .withColumn("_processing_layer", F.lit("GOLD"))
    .withColumn("_record_status", F.lit("AI_READY"))
)

In [ ]:
gold_columns = [
    "CustomerId",
    "Surname",
    "CreditScore",
    "CreditScoreBand",
    "LowCreditFlag",
    "Geography",
    "GeographyCode",
    "Gender",
    "GenderCode",
    "Age",
    "AgeGroup",
    "Tenure",
    "TenureGroup",
    "Balance",
    "BalanceBand",
    "EstimatedSalary",
    "SalaryBand",
    "BalanceToSalaryRatio",
    "ZeroBalanceFlag",
    "NumOfProducts",
    "ProductEngagementLevel",
    "HasCrCard",
    "IsActiveMember",
    "InactiveMemberFlag",
    "EngagementScore",
    "CustomerValueScore",
    "CustomerValueSegment",
    "BehavioralRiskScore",
    "BehavioralRiskBand",
    "Exited",
    "_experiment_id",
    "_experiment_seed",
    "_silver_processed_at",
    "_quality_run_id",
    "_gold_run_id",
    "_gold_processed_at",
    "_processing_layer",
    "_record_status"
]

gold_df = gold_df.select(*gold_columns)

print(f"Gold rows: {gold_df.count()}")
print(f"Gold columns: {len(gold_df.columns)}")

display(gold_df.limit(20))

In [ ]:
gold_validation_df = gold_df.agg(
    F.count("*").alias("total_rows"),
    F.countDistinct("CustomerId").alias("distinct_customers"),

    F.sum(
        F.col("AgeGroup").isNull().cast("int")
    ).alias("null_age_groups"),

    F.sum(
        F.col("CreditScoreBand").isNull().cast("int")
    ).alias("null_credit_bands"),

    F.sum(
        F.col("BalanceBand").isNull().cast("int")
    ).alias("null_balance_bands"),

    F.sum(
        F.col("SalaryBand").isNull().cast("int")
    ).alias("null_salary_bands"),

    F.sum(
        F.col("CustomerValueSegment").isNull().cast("int")
    ).alias("null_value_segments"),

    F.sum(
        F.col("BehavioralRiskBand").isNull().cast("int")
    ).alias("null_risk_bands"),

    F.sum(
        F.col("BalanceToSalaryRatio").isNull().cast("int")
    ).alias("null_balance_salary_ratios")
)

display(gold_validation_df)

In [ ]:
from pyspark.sql.window import Window

churn_distribution_df = (
    gold_df
    .groupBy("Exited")
    .agg(
        F.count("*").alias("customer_count")
    )
    .withColumn(
        "percentage",
        F.round(
            F.col("customer_count")
            / F.sum("customer_count").over(Window.partitionBy())
            * 100,
            2
        )
    )
    .orderBy("Exited")
)

display(churn_distribution_df)

In [ ]:
gold_summary_df = (
    gold_df
    .groupBy(
        "Geography",
        "AgeGroup",
        "Gender",
        "CreditScoreBand",
        "BehavioralRiskBand"
    )
    .agg(
        F.count("*").alias("customer_count"),
        F.sum("Exited").alias("churned_customers"),
        F.round(F.avg("Exited") * 100, 2).alias("churn_rate_percent"),
        F.round(F.avg("CreditScore"), 2).alias("average_credit_score"),
        F.round(F.avg("Age"), 2).alias("average_age"),
        F.round(F.avg("Balance"), 2).alias("average_balance"),
        F.round(
            F.avg("EstimatedSalary"),
            2
        ).alias("average_estimated_salary"),
        F.round(
            F.avg("CustomerValueScore"),
            2
        ).alias("average_customer_value_score"),
        F.round(
            F.avg("EngagementScore"),
            2
        ).alias("average_engagement_score")
    )
    .withColumn("_gold_run_id", F.lit(GOLD_RUN_ID))
    .withColumn("_generated_at", F.current_timestamp())
)
display(
    gold_summary_df
    .orderBy(F.desc("churn_rate_percent"))
    .limit(30)
)

In [ ]:
start_time = time.perf_counter()

measured_gold_row_count = gold_df.count()

end_time = time.perf_counter()

gold_transformation_seconds = end_time - start_time

print(f"Measured Gold rows: {measured_gold_row_count}")
print(
    f"Gold transformation action time: "
    f"{gold_transformation_seconds:.4f} seconds"
)

In [ ]:
(
    gold_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_TABLE)
)

print(f"Gold customer table created: {GOLD_TABLE}")

In [ ]:
(
    gold_summary_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_SUMMARY_TABLE)
)

print(f"Gold summary table created: {GOLD_SUMMARY_TABLE}")

In [ ]:
gold_feature_count = len(gold_df.columns)

gold_metrics_data = [
    ("gold_row_count", float(measured_gold_row_count)),
    ("gold_column_count", float(gold_feature_count)),
    ("gold_distinct_customers", float(
        gold_df.select("CustomerId").distinct().count()
    )),
    ("gold_null_engineered_features", float(
        gold_df.filter(
            F.col("AgeGroup").isNull()
            | F.col("CreditScoreBand").isNull()
            | F.col("BalanceBand").isNull()
            | F.col("SalaryBand").isNull()
            | F.col("CustomerValueSegment").isNull()
            | F.col("BehavioralRiskBand").isNull()
        ).count()
    )),
    ("gold_transformation_seconds", float(
        gold_transformation_seconds
    ))
]

gold_metrics_df = (
    spark.createDataFrame(
        gold_metrics_data,
        ["metric_name", "metric_value"]
    )
    .withColumn("experiment_stage", F.lit("GOLD_AI_READY"))
    .withColumn(
        "dataset_name",
        F.lit("gold_customer_churn")
    )
    .withColumn("ingestion_batch_id", F.lit(GOLD_RUN_ID))
    .withColumn("measured_at", F.current_timestamp())
)

display(gold_metrics_df)

In [ ]:
(
    gold_metrics_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(METRICS_TABLE)
)

print(f"Gold metrics appended to: {METRICS_TABLE}")

In [ ]:
spark.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT CustomerId) AS distinct_customers,
    COUNT(*) - COUNT(DISTINCT CustomerId) AS duplicate_customers,
    SUM(CASE WHEN AgeGroup IS NULL THEN 1 ELSE 0 END)
        AS null_age_groups,
    SUM(CASE WHEN BehavioralRiskBand IS NULL THEN 1 ELSE 0 END)
        AS null_risk_bands
FROM {GOLD_TABLE}
""").show(truncate=False)

spark.sql(f"""
SELECT
    COUNT(*) AS summary_segments,
    SUM(customer_count) AS represented_customers,
    MIN(churn_rate_percent) AS minimum_churn_rate,
    MAX(churn_rate_percent) AS maximum_churn_rate
FROM {GOLD_SUMMARY_TABLE}
""").show(truncate=False)

In [ ]:
display(
    spark.sql(f"DESCRIBE HISTORY {GOLD_TABLE}")
)

display(
    spark.sql(f"DESCRIBE HISTORY {GOLD_SUMMARY_TABLE}")
)

In [ ]:
display(
    gold_df
    .groupBy("Geography")
    .agg(
        F.count("*").alias("customer_count"),
        F.round(F.avg("Exited") * 100, 2)
         .alias("churn_rate_percent")
    )
    .orderBy(F.desc("churn_rate_percent"))
)

In [ ]:
display(
    gold_df
    .groupBy("AgeGroup")
    .agg(
        F.count("*").alias("customer_count"),
        F.round(F.avg("Exited") * 100, 2)
         .alias("churn_rate_percent")
    )
    .orderBy("AgeGroup")
)

In [ ]:
display(
    gold_df
    .groupBy("BehavioralRiskBand")
    .agg(
        F.count("*").alias("customer_count"),
        F.round(F.avg("Exited") * 100, 2)
         .alias("churn_rate_percent")
    )
    .orderBy("BehavioralRiskBand")
)

In [ ]:
display(
    gold_df
    .groupBy("CustomerValueSegment")
    .agg(
        F.count("*").alias("customer_count"),
        F.round(F.avg("Exited") * 100, 2)
         .alias("churn_rate_percent"),
        F.round(F.avg("Balance"), 2)
         .alias("average_balance")
    )
    .orderBy("CustomerValueSegment")
)